<a href="https://colab.research.google.com/github/jandorrvfx-code/tennis_analysis/blob/main/training/tennis_court_keypoints_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download dataset using gdown

In [1]:
# Install gdown for reliable Google Drive downloads
!pip install gdown -q

In [2]:
# Download the dataset using gdown with the correct file ID
# The file ID '1lhAaeQCmk2y440PmagA0KmIVBIysVMwu' is extracted from the original wget URL.
!gdown 1lhAaeQCmk2y440PmagA0KmIVBIysVMwu -O tennis_court_det_dataset.zip

Downloading...
From (original): https://drive.google.com/uc?id=1lhAaeQCmk2y440PmagA0KmIVBIysVMwu
From (redirected): https://drive.google.com/uc?id=1lhAaeQCmk2y440PmagA0KmIVBIysVMwu&confirm=t&uuid=24089f76-0639-4a3a-a81a-5eb0b98e9905
To: /content/tennis_court_det_dataset.zip
100% 7.26G/7.26G [02:04<00:00, 58.4MB/s]


In [3]:
# Unzip the newly downloaded dataset
!unzip -q tennis_court_det_dataset.zip
print('Dataset unzipped successfully.')

Dataset unzipped successfully.


# Start code

In [13]:
from google.colab import files

# Ensure the model file exists before attempting to download
if os.path.exists(model_path):
  files.download(model_path)
else:
  print(f"Error: Model file not found at {model_path}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Alternatively, you can navigate to the file browser on the left-hand side of Colab, locate the `models` directory, and then right-click on `keypoints_model.pth` to download it.

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import json
import cv2
import numpy as np
import random

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Create Torch Dataset

In [6]:
from IPython.core.display import json
class KeypointsDataset(Dataset):
  def __init__(self, img_dir, data_file):
    self.img_dir = img_dir
    with open(data_file, "r") as f:
      self.data = json.load(f)

    self.transforms = transforms.Compose([
        transforms.ToPILImage(),  # PIL image is an object that holds and manipulates an image
        transforms.Resize((224, 224)),
        transforms.ToTensor(),  # Raw numerical representation of an image
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalize values so the model understands
    ])

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    "Return image and keypoints"
    item = self.data[idx]
    img = cv2.imread(f"{self.img_dir}/{item['id']}.png")
    h, w  = img.shape[:2]

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # BGR to RGB
    img = self.transforms(img)  # Apply transforms
    kps = np.array(item['kps']).flatten()  # Convert keypoints from nD to 1D
    kps = kps.astype(np.float32)

    # Adjust keypoints to resized image
    kps[::2] *= 224.0 / w  # Adjust x coordinates
    kps[1::2] *= 224.0 / h  # Adjust y coordinates

    return img, kps




In [7]:
train_dataset = KeypointsDataset("data/images", "data/data_train.json")  # Train dataset
val_dataset = KeypointsDataset("data/images", "data/data_val.json")  # Validation dataset

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
var_loader = DataLoader(val_dataset, batch_size=8, shuffle=True)

# Create model

In [8]:
model = models.resnet50(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, 14*2)  # Replaces the last layer of the neural network

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 216MB/s]


In [9]:
model = model.to(device)

# Train model

In [10]:
criterion = torch.nn.MSELoss()  # Define loss function
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)  # lr = learning rate

In [11]:
num_epochs=20
for epoch in range(num_epochs):
  for i, (imgs, kps) in enumerate(train_loader):
    imgs = imgs.to(device)
    kps = kps.to(device)

    optimizer.zero_grad()  # Flush optimizer gradients
    outputs = model(imgs)  # Have model predict the keypoints on the image
    loss = criterion(outputs, kps)  # Calculate loss
    loss.backward()  # Backward propagation
    optimizer.step()  # Do a learning step

    if i % 10 == 0:  # Log every 10 steps. If i is divisible by 10
      print(f"Epoch {epoch+1}/{num_epochs}, Step {i+1}/{len(train_loader)}, Loss: {loss.item()}")

Epoch 1/20, Step 1/829, Loss: 14598.66015625
Epoch 1/20, Step 11/829, Loss: 14359.6337890625
Epoch 1/20, Step 21/829, Loss: 14866.509765625
Epoch 1/20, Step 31/829, Loss: 13831.0068359375
Epoch 1/20, Step 41/829, Loss: 13943.060546875
Epoch 1/20, Step 51/829, Loss: 12734.0927734375
Epoch 1/20, Step 61/829, Loss: 13812.1103515625
Epoch 1/20, Step 71/829, Loss: 12260.2978515625
Epoch 1/20, Step 81/829, Loss: 11984.1181640625
Epoch 1/20, Step 91/829, Loss: 11167.6005859375
Epoch 1/20, Step 101/829, Loss: 10717.7841796875
Epoch 1/20, Step 111/829, Loss: 9980.19921875
Epoch 1/20, Step 121/829, Loss: 10218.8818359375
Epoch 1/20, Step 131/829, Loss: 10387.1435546875
Epoch 1/20, Step 141/829, Loss: 10039.875
Epoch 1/20, Step 151/829, Loss: 8959.7578125
Epoch 1/20, Step 161/829, Loss: 9050.841796875
Epoch 1/20, Step 171/829, Loss: 8452.322265625
Epoch 1/20, Step 181/829, Loss: 8886.494140625
Epoch 1/20, Step 191/829, Loss: 8339.564453125
Epoch 1/20, Step 201/829, Loss: 7449.07275390625
Epoch 1/

# Save model

In [14]:
# Save the model weights
import os

# Ensure the models directory exists
models_dir = './'
os.makedirs(models_dir, exist_ok=True)

# Save the model
model_path = os.path.join(models_dir, 'keypoints_model.pth')
torch.save(model.state_dict(), model_path)

print(f"Model saved successfully to: {model_path}")


Model saved successfully to: ./keypoints_model.pth
